In [1]:
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer
import pandas as pd
from datasets import load_dataset
from transformers import pipeline
from accelerate import Accelerator

W0222 16:08:52.100000 412 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [2]:
device = Accelerator().device
device

device(type='cpu')

In [3]:
def format_chat_template(sample):
    return {
        'messages': [
            {'role': 'user', 'content': sample['instruction']},
            {'role': 'assistant', 'content': sample['response']}
        ]
    }

In [6]:
dataset_path = r'../../data/train_data/messages_intr.csv'
data = load_dataset("csv", data_files=dataset_path, split="train")
dataset = data.map(format_chat_template, remove_columns=['instruction', 'response'])

In [7]:
model_id = r'google/gemma-3-270m-it'
dataset_path = r'../../data/train_data/messages_intr.csv'

In [8]:
model = AutoModelForCausalLM.from_pretrained(model_id, device_map='auto', torch_dtype='auto')
tokenizer = AutoTokenizer.from_pretrained(model_id)

`torch_dtype` is deprecated! Use `dtype` instead!


In [9]:
tokenizer.chat_template

'{{ bos_token }}\n{%- if messages[0][\'role\'] == \'system\' -%}\n    {%- if messages[0][\'content\'] is string -%}\n        {%- set first_user_prefix = messages[0][\'content\'] + \'\n\n\' -%}\n    {%- else -%}\n        {%- set first_user_prefix = messages[0][\'content\'][0][\'text\'] + \'\n\n\' -%}\n    {%- endif -%}\n    {%- set loop_messages = messages[1:] -%}\n{%- else -%}\n    {%- set first_user_prefix = "" -%}\n    {%- set loop_messages = messages -%}\n{%- endif -%}\n{%- for message in loop_messages -%}\n    {%- if (message[\'role\'] == \'user\') != (loop.index0 % 2 == 0) -%}\n        {{ raise_exception("Conversation roles must alternate user/assistant/user/assistant/...") }}\n    {%- endif -%}\n    {%- if (message[\'role\'] == \'assistant\') -%}\n        {%- set role = "model" -%}\n    {%- else -%}\n        {%- set role = message[\'role\'] -%}\n    {%- endif -%}\n    {{ \'<start_of_turn>\' + role + \'\n\' + (first_user_prefix if loop.first else "") }}\n    {%- if message[\'conte

In [10]:
pipe = pipeline('text-generation', model=model, tokenizer=tokenizer)

Device set to use cpu


In [31]:
system_prompt = """Ты — лингвист-аналитик. Твоя задача: найти общий смысл в сообщениях.
ПРАВИЛА:
1. Ответ строго в формате: [обобщённый смысл]
2. Никаких лишних слов, пояснений и вежливости.
3. Не пересказывай детали.
4. Обобщай людей до понятий (личность, знакомые, коллектив).
5. Будь максимально кратким."""

In [32]:
new_msg = """1) Я соскучилась по Мусе
2) Все скучали
3) Прям скучаю
4) Маракасики я по вам скучаю
5) Все, кого не было, пропустили очень много: страшилки, брайнрот, игры, шлепанье, ножки
"""

In [53]:
message = [
    {'role': 'system', 'content': system_prompt}
]

for item in dataset:
    message.append(
        {'role': 'user', 'content': f"Проанализируй эти сообщения и выведи общую тему:\n\n{item['messages'][0]['content']}"}
    )
    message.append(
        {'role': 'assistant', 'content': item['messages'][1]['content']}
    )
message.append(
    {'role': 'user', 'content': new_msg}
)

In [54]:
output = pipe(message)

In [56]:
output[0]['generated_text'][-1]['content']

'Надо было\n'

In [4]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.1,
    bias='none',
    task_type='CAUSAL_LM'
)

In [10]:
from trl import SFTTrainer, SFTConfig

In [13]:
torch_dtype = model.dtype

learning_rate = 1e-5

args = SFTConfig(
    output_dir=checkpoint_dir,              # directory to save and repository id
    max_length=512,                         # max sequence length for model and packing of the dataset
    packing=False,                          # Groups multiple samples in the dataset into a single sequence
    num_train_epochs=5,                     # number of training epochs
    per_device_train_batch_size=4,          # batch size per device during training
    gradient_checkpointing=False,           # Caching is incompatible with gradient checkpointing
    optim="adamw_torch_fused",              # use fused adamw optimizer
    logging_steps=1,                        # log every step
    learning_rate=learning_rate,            # learning rate
    fp16=True if torch_dtype == torch.float16 else False,   # use float16 precision
    bf16=True if torch_dtype == torch.bfloat16 else False,  # use bfloat16 precision
    lr_scheduler_type="constant",           # use constant learning rate scheduler
    dataset_kwargs={
        "add_special_tokens": False, # Template with special tokens
        "append_concat_token": True, # Add EOS token as separator token between examples
    }
)
    
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=args,
    processing_class=tokenizer
)

Tokenizing train dataset:   0%|          | 0/3 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/3 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.


In [14]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 1}.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\Denis\_netrc.
wandb: Currently logged in as: salnikovdenis44 (salnikovdenis44-omstu) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


C:\Users\Denis\miniconda3\envs\myenv\lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


TrainOutput(global_step=3, training_loss=5.480406443277995, metrics={'train_runtime': 23.3178, 'train_samples_per_second': 0.386, 'train_steps_per_second': 0.129, 'total_flos': 690015456000.0, 'train_loss': 5.480406443277995, 'entropy': 2.854021151860555, 'num_tokens': 939.0, 'mean_token_accuracy': 0.3655913968880971, 'epoch': 3.0})

In [ ]:
save_dir = r'./'
trainer.save_model()

In [ ]:
import matplotlib.pyplot as plt

log_history = trainer.state.log_history

train_losses = [log["loss"] for log in log_history if "loss" in log]
epoch_train = [log["epoch"] for log in log_history if "loss" in log]
eval_losses = [log["eval_loss"] for log in log_history if "eval_loss" in log]
epoch_eval = [log["epoch"] for log in log_history if "eval_loss" in log]

# Plot the training loss
plt.plot(epoch_train, train_losses, label="Training Loss")
plt.plot(epoch_eval, eval_losses, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss per Epoch")
plt.legend()
plt.grid(True)
plt.show()